# 03 — Temporal Handling: Point-Based vs Sliding-Window Aggregation

**Goal (bonus temporal):** justify feature-based windowing per Keogh & Lin (2005) — raw subsequence K-Means yields meaningless sinusoidal centroids.
Window default 60s / step 30s, features mean+std+delta per sensor (8x3=24 dims).

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

SENSOR_8=['Rate Of Penetration (ft_per_hr)','Weight on Bit (klbs)','Rotary RPM (RPM)','Standpipe Pressure (psi)','Rotary Torque (kft_lb)','Hook Load (klbs)','Total Pump Output (gal_per_min)','Differential Pressure (psi)']
CSV_NAME='56-32 1sec data 27029986.csv'
W, STEP = 60, 30

def resolve_input(csv=CSV_NAME):
    root = Path('/kaggle/input')
    if root.exists():
        hits = list(root.rglob(csv))
        if hits:
            return hits[0]
        for p in root.rglob('*.csv'):
            if '56-32' in p.name or '1sec' in p.name or '27029986' in p.name:
                return p
        csvs = [p for p in root.rglob('*.csv') if p.is_file()]
        if csvs:
            return max(csvs, key=lambda p: p.stat().st_size)
        pqs = [p for p in root.rglob('*.parquet') if p.is_file()]
        if pqs:
            return max(pqs, key=lambda p: p.stat().st_size)
    for c in [Path('dataset')/csv, Path('../dataset')/csv, Path.cwd()/'dataset'/csv]:
        if c.exists(): return c
    listing = [str(p) for p in sorted(root.rglob('*'))][:30] if root.exists() else ['</kaggle/input not found>']
    raise FileNotFoundError('Dataset CSV not found. /kaggle/input contains: ' + str(listing) + ' | Fix: right panel Add Input -> add your private dataset -> re-run.')
if Path('/kaggle/working').exists(): OUT = Path('/kaggle/working')  # live session: visible in file browser, downloadable, saved on Save Version
elif Path('/kaggle/output').exists(): OUT = Path('/kaggle/output')
else: OUT = Path('../figures')
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
usecols=['YYYY/MM/DD','HH:MM:SS']+SENSOR_8
df=pd.read_csv(resolve_input(), usecols=usecols, nrows=30000, low_memory=True)
for c in SENSOR_8: df[c]=pd.to_numeric(df[c], errors='coerce').astype('float32')
df=df.replace(-999.25, np.nan).dropna().reset_index(drop=True)
df['datetime']=pd.to_datetime(df['YYYY/MM/DD']+' '+df['HH:MM:SS'], format='%Y/%m/%d %H:%M:%S', errors='coerce')
df=df.sort_values('datetime').reset_index(drop=True)
print(df.shape, df['datetime'].min(), '->', df['datetime'].max())

In [ ]:
rows=[]
for s in range(0, len(df)-W+1, STEP):
    ch=df.iloc[s:s+W]; rec={'t0':ch['datetime'].iloc[0]}
    for c in SENSOR_8:
        v=ch[c].to_numpy(dtype='float64')
        rec[c+'__mean']=v.mean(); rec[c+'__std']=v.std(); rec[c+'__delta']=v[-1]-v[0]
    rows.append(rec)
win=pd.DataFrame(rows)
print('windows:', win.shape, '(24 feature cols)')
display(win.iloc[:2, :5])

In [ ]:
Xp=RobustScaler().fit_transform(df[SENSOR_8].sample(min(15000,len(df)), random_state=42))
lp=KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(Xp)
feat=[c for c in win.columns if c!='t0']
Xw=RobustScaler().fit_transform(win[feat].to_numpy())
lw=KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(Xw)
print(f"point-based silhouette: {silhouette_score(Xp,lp):.4f}")
print(f"window-based silhouette: {silhouette_score(Xw,lw):.4f}")
pd.DataFrame({'representation':['point','window-60s'],'silhouette':[silhouette_score(Xp,lp), silhouette_score(Xw,lw)]}).to_csv(OUT/'table3_temporal_compare.csv', index=False)

In [ ]:
plt.figure(figsize=(10,3))
plt.plot(df['datetime'].iloc[:3000], df['Standpipe Pressure (psi)'].iloc[:3000], lw=0.7)
plt.title('Standpipe Pressure (psi) — first 3000s: temporal dependency is visible')
plt.xlabel('Time'); plt.ylabel('psi')
plt.tight_layout(); plt.savefig(OUT/'fig4_temporal_trace.png', dpi=150); plt.show()

#### Cell: Export & verify — collect everything for download
**What:** lists what was actually written to `OUT`, copies it into `/kaggle/working` (the live, downloadable folder), and packs it into one `kaggle_outputs.zip`.
**Why:** if this cell reports `<EMPTY>`, the cells above never ran (error or skipped) — outputs cannot exist. The zip gives you one-click download instead of per-file clicks.
**Read this:** the printed file list (expect `table*.csv` + `fig*.png`) and the zip size; then download via the file browser.


In [ ]:
from pathlib import Path
import shutil, zipfile
OUT.mkdir(parents=True, exist_ok=True)
live = Path('/kaggle/working') if Path('/kaggle/working').exists() else OUT
moved = []
for f in sorted(OUT.glob('*')):
    if not f.is_file() or f.suffix == '.zip':
        continue
    t = live / f.name
    if f.resolve() != t.resolve():
        shutil.copy2(f, t)
    moved.append(f.name)
print('OUT =', OUT)
print('files:', moved if moved else '<EMPTY — cells above did not run or errored, re-run from the top>')
zp = live / 'kaggle_outputs.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in moved:
        z.write(live / name, name)
print(f'zip: {zp} ({zp.stat().st_size/1024:.1f} KB)')
print('Download: file browser ->', live, '-> kaggle_outputs.zip. For the paper link: Save Version -> Output tab.')

try:
    from IPython.display import FileLink, display as _display
    _display(FileLink(str(zp)))
    print('Click the link above to download the zip directly (no file browser needed).')
except Exception as _e:
    print('direct-link unavailable:', _e)
